<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/07_3_Memory_QA_Agent_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 07-3: Memory 기반 Q&A 에이전트 구현
이 노트북에서는 LangChain의 가장 핵심적인 단위인 **Chain**을 구성하고, 이를 통해 질의 내용을 기억하는 메모리 기반 질의응답(Q&A) 에이전트를 만드는 실습을 진행합니다.

### 1. 환경 준비
필요한 라이브러리를 설치합니다.

In [19]:
# langchain 최신 버전으로 강제 설치합니다.
!pip install -q -U --force-reinstall langchain langchain-community langchain-huggingface langchain-core langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.1.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.1 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.1.0 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.1 wh

### 2. Gemini LLM 로드

Google Gemini 모델을 사용하여 더 나은 답변을 시도할 수 있습니다. Gemini 모델을 사용하려면 `google-generativeai` 라이브러리를 설치하고 API 키를 설정해야 합니다.

Gemini API를 사용하려면 API 키가 필요합니다.

아직 키가 없다면 Google AI Studio에서 키를 생성하세요.

1. Google AI Studio 접속
먼저 공식 사이트(https://aistudio.google.com)에 접속합니다. 사용 중인 구글 계정으로 로그인해 주세요.

2. 서비스 약관 동의
처음 접속하신 경우, 생성형 AI 사용을 위한 서비스 약관 동의 팝업이 뜹니다. 내용을 확인하신 후 'Accept' 또는 'Continue' 버튼을 클릭하여 메인 대시보드로 진입합니다.

3. API 키 메뉴 이동  
    - [대시보드] 왼쪽 상단 메뉴 바에서 [Get API key] 항목을 클릭합니다.

4. API 키 생성: 화면 중앙에 보이는 버튼 중 하나를 선택합니다.  

    - [Create API key] in new project: 새로운 프로젝트를 생성하면서 키를 발급받습니다. (처음 만드시는 분들께 권장)

    - Create API key in existing project: 기존에 사용하던 Google Cloud 프로젝트가 있다면 해당 프로젝트를 선택하여 키를 생성합니다.

5. 키 복사 및 안전한 보관:  
팝업창에 생성된 **긴 문자열(API Key)**이 나타납니다. 'Copy' 버튼을 눌러 복사한 뒤, 메모장이나 환경 변수 설정 등 안전한 곳에 저장해 두세요.

    ⚠️ 주의: API 키는 비밀번호와 같습니다. GitHub 같은 공개 저장소에 코드를 올릴 때 키가 노출되지 않도록 주의하세요!

6. 팁: 요금 및 제한 사항 (무료 티어 기준)  

    - Gemini 1.5 Flash: 속도가 빠르고 무료 사용량이 넉넉하여 테스트용으로 좋습니다.  
    - Gemini 1.5 Pro: 복잡한 추론에 적합하지만, 무료 티어에서는 분당 요청 횟수(RPM) 제한이 더 타이트합니다.  
    - 개인정보: 무료 등급 사용 시 입력한 데이터는 모델 학습에 사용될 수 있으므로 민감한 정보는 입력하지 않는 것이 좋습니다.  

Colab에서는 왼쪽 패널의 "🔑" 아래에 키를 `GOOGLE_API_KEY`라는 이름으로 Secrets Manager에 추가하세요. 그런 다음 키를 SDK에 전달합니다.

In [20]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

Gemini API 설정 완료


3. LCEL 기반 Memory QA 에이전트 구현  

- 이 코드는 LLMChain을 대신하여 Runnable 인터페이스를 통해 메모리와 프롬프트를 결합합니다.  

- 기존 LangChain의 오류 해결을 위한 해결책  
    - 모듈 직접 타격: 에러가 계속 발생하는 langchain.memory 모듈 대신, 더 하위 레벨에서 안정적으로 작동하는 langchain_community.chat_message_histories를 사용했습니다.  
    - 의존성 탈피: LLMChain 같은 레거시 모듈을 완전히 버리고 최신 LCEL(|) 구조를 채택하여 패키지 업데이트에 따른 오류 가능성을 차단했습니다.
    - 수동 제어: 자동 메모리 주입 과정에서 발생하는 오류를 막기 위해, 대화 기록을 직접 문자열로 추출하여 프롬프트에 넣는 방식을 사용하여 가독성과 안정성을 모두 잡았습니다.  

In [21]:
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# --- [해결책] 에러가 잦은 memory 모듈 대신 community의 기록 저장소 활용 ---
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. 대화 기록을 담을 객체 생성 (메모리 모듈 오류 우회)
history_db = ChatMessageHistory()

# 2. Gemini LLM 설정
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY) # 모델 이름을 'gemini-flash-latest'로 변경

# 3. 프롬프트 정의
template = """너는 사용자의 이름을 기억하고 이전 대화를 참고하는 똑똑한 AI야.
아래 대화 기록을 바탕으로 질문에 답해줘.

[대화 기록]
{history}

질문: {question}
답변:"""

prompt = PromptTemplate.from_template(template)

# 4. LCEL 데이터 흐름 정의
def get_history_string(_):
    # 저장된 메시지들을 문자열로 변환하여 반환합니다.
    messages = history_db.messages
    return "\n".join([f"{'Human' if i%2==0 else 'AI'}: {m.content}" for i, m in enumerate(messages)])

lcel_chain = (
    {
        "question": RunnablePassthrough(),
        "history": RunnableLambda(get_history_string)
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 5. 실행 및 기록 수동 저장 함수
def chat(user_input):
    # 답변 생성
    response = lcel_chain.invoke(user_input)

    # 대화 기록을 history_db에 직접 추가 (Stateful 유지)
    history_db.add_user_message(user_input)
    history_db.add_ai_message(response)

    return response

print("에러 없이 에이전트가 로드되었습니다! 이제 chat('질문')을 실행해 보세요.")

에러 없이 에이전트가 로드되었습니다! 이제 chat('질문')을 실행해 보세요.


4. 실습 테스트 (지시어 처리 확인)  
- 에이전트가 이전 대화의 정보를 정확히 기억하고 활용하는지 확인합니다.  

In [18]:
# --- [테스트 시나리오] ---

# 1. 정보 주입 (사용자 정보 전달)
print("질문 1: 안녕! 나는 서울에 살고 있는 제미니라고 해. 요즘 랭체인을 배우고 있어.")
response1 = chat("안녕! 나는 서울에 살고 있는 제미니라고 해. 요즘 랭체인을 배우고 있어.")
print(f"AI 답변: {response1}\n")

# 2. 기억 확인 (이름과 지역 확인)
print("질문 2: 내 이름이 뭐라고 했지? 그리고 내가 어디 산다고 했어?")
response2 = chat("내 이름이 뭐라고 했지? 그리고 내가 어디 산다고 했어?")
print(f"AI 답변: {response2}\n")

# 3. 복합 문맥 확인 (지시어 처리)
print("질문 3: 내가 지금 배우고 있는 '그것'의 장점 하나만 알려줘.")
response3 = chat("내가 지금 배우고 있는 '그것'의 장점 하나만 알려줘.")
print(f"AI 답변: {response3}\n")

질문 1: 안녕! 나는 서울에 살고 있는 제미니라고 해. 요즘 랭체인을 배우고 있어.
AI 답변: 안녕하세요, 제미니 님! 서울에 살고 계시는군요. 만나 뵙게 되어 반갑습니다.

요즘 인공지능 분야에서 가장 주목받는 기술 중 하나인 랭체인(LangChain)을 배우고 계시다니 정말 멋지네요! 랭체인 학습은 잘 되고 계신가요? 혹시 궁금한 점이나 도움이 필요하신 부분이 있다면 언제든지 저에게 물어봐 주세요!

질문 2: 내 이름이 뭐라고 했지? 그리고 내가 어디 산다고 했어?
AI 답변: 사용자님의 이름은 **제미니**이고, **서울**에 살고 계십니다.

최근에 배우고 계신 랭체인 학습은 여전히 잘 진행되고 계신가요?

질문 3: 내가 지금 배우고 있는 '그것'의 장점 하나만 알려줘.
AI 답변: 사용자님께서 현재 배우고 계신 '그것'은 **랭체인(LangChain)**입니다.

랭체인(LangChain)의 장점 하나는 **대규모 언어 모델(LLM)을 활용하여 데이터 검색, 외부 API 호출 등 여러 단계를 거치는 복잡한 애플리케이션(체인 및 에이전트)을 쉽고 효율적으로 구축할 수 있도록 모듈화된 구성 요소를 제공**한다는 점입니다.

